# Privacy-First Curation of Agentic Trajectories

**End-to-end pipeline walkthrough:** synthetic wearable data → multi-persona annotation → IRR calibration → process reward labeling → annotator poisoning detection

**Audience:** Staff engineers at Anthropic, OpenAI, Cohere, DeepMind, Writer, AI21, Kore.ai  
**Repo:** `llm-wearable-agentic-eval-pipeline` · Day 18 of 45-day sprint  
**All numbers are from real pipeline runs (dry-run mode). Post-calibration κ = 1.0 is a dry-run artifact; live-API expected range κ ≈ 0.55–0.65 → 0.78–0.85.**

---
## Section 1 — Overview & Motivation

### The Three Problems This Pipeline Solves

**Problem 1 — Outcome Reward Models (ORM) destroy step-level signal.**  
ORM assigns reward only at the terminal step. If step 15 fails, all 14 correct intermediate steps are penalized identically to a model that failed from step 1. This *gradient conflict* problem is quantified in ReasonRAG (NeurIPS 2025, arXiv 2505.14069): process-supervised training is **18× more data-efficient** than outcome-supervised RL, because each intermediate step carries signal instead of being zeroed out.

**Problem 2 — Annotation quality is unmeasured.**  
Cohere Command A (arXiv 2504.00698) describes 800-prompt annotation across 65 annotators — but reports **zero agreement statistics** (no κ, no α). Kore.ai (Oct 2025) found 89% of enterprises track agent observability, but only 52% have *real evaluation*. The gap is methodology, not tooling. Standard inter-rater reliability (IRR) breaks for non-deterministic agents: two valid paths to the same goal register as *disagreement*, not agreement.

**Problem 3 — Annotator poisoning is harder to detect than assumed.**  
Anthropic (Oct 2025) showed that **only 250 malicious documents** (0.00016% of training data) suffice to backdoor models of any size. Count-based, not proportion-based. Our detector exposes a named blind spot: three colluding annotators with identical bias collapse to panel consensus under MAD scoring, yielding suspicion = 0.0 — indistinguishable from the most reliable annotator.

### Pipeline Architecture

```
synthetic_wearable_logs.jsonl (100 logs)
    │
    ▼
annotator_simulator.py  →  day12_annotations.jsonl  (25 records, 5 personas × 5 logs)
    │                 →  annotations_round2.json    (150 records, 5 personas × 30 logs, calibrated)
    ▼
pia_calculator.py       →  pia_results.json         (Δκ = +0.808, standard→PIA)
    │
    ▼
prm_annotator.py        →  prm_annotations_20.jsonl (60 step rewards across 20 trajectories)
    │
    ▼
poisoning_detector.py   →  day17_detection_results.json (MAD blind spot: F1 = 0.0)
```

In [ ]:
"""Notebook setup: paths, imports, matplotlib style."""
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# ── repo root ──────────────────────────────────────────────────────────────
REPO_ROOT = Path("__file__").resolve().parent.parent
# When running interactively, __file__ is not defined; fall back to cwd.
try:
    REPO_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_ANN = REPO_ROOT / "data" / "annotations"
DATA_PROC = REPO_ROOT / "data" / "processed"
FIG_DIR = Path(__file__).resolve().parent / "figures" if False else (
    REPO_ROOT / "notebooks" / "figures"
)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── matplotlib style ───────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

PALETTE = {
    "good": "#2ecc71",
    "warn": "#f39c12",
    "bad": "#e74c3c",
    "neutral": "#3498db",
    "dark": "#2c3e50",
}

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"FIG_DIR:   {FIG_DIR}")

---
## Section 2 — Synthetic Data Generation

We generate privacy-preserving wearable sensor logs using `src/data/wearable_generator.py`. Each log captures a realistic scenario (health alert, privacy-sensitive context, location trigger, ambient noise, or calendar reminder) with differential privacy applied: Gaussian mechanism at ε = 1.0, σ ≈ 48 bpm for heart rate.

Key implementation note from CLAUDE.md: noised sensor values **must not be bounded to physiological ranges** at ε = 1.0. Check `math.isfinite`, not physiological validity.

In [ ]:
"""Load synthetic wearable logs and display one sample record."""
logs_path = DATA_RAW / "synthetic_wearable_logs.jsonl"

if not logs_path.exists():
    print(f"TODO: Generate synthetic data first.\n"
          f"Run: uv run python -m src.data.wearable_generator "
          f"--count 100 --output {logs_path} --seed 42")
    logs: list[dict] = []
else:
    logs = [json.loads(line) for line in logs_path.read_text().splitlines() if line.strip()]
    print(f"Loaded {len(logs)} synthetic wearable logs.")
    print(f"\n── Sample record (log_id: {logs[0]['log_id']}) ──")
    sample = logs[0]
    print(f"  scenario_type : {sample['scenario_type']}")
    print(f"  consent_model : {sample['consent_model']}")
    sd = sample["sensor_data"]
    print(f"  heart_rate    : {sd['heart_rate']:.1f} bpm  (noised: {sd['heart_rate_noised']:.1f})")
    print(f"  spo2          : {sd['spo2']:.1f}%    (noised: {sd['spo2_noised']:.1f})")
    print(f"  gps           : ({sd['gps_lat']:.5f}, {sd['gps_lon']:.5f})")
    print(f"  trajectory steps: {len(sample['trajectory'])}")
    for step in sample["trajectory"]:
        print(f"    [{step['step_index']}] {step['step_name']}: action={step['action'] or '(none)'}")

In [ ]:
"""Chart: scenario type distribution across the synthetic dataset."""
if not logs:
    print("No logs loaded — skipping chart.")
else:
    scenario_counts = Counter(log["scenario_type"] for log in logs)
    scenarios = sorted(scenario_counts.keys())
    counts = [scenario_counts[s] for s in scenarios]

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(
        [s.replace("_", "\n") for s in scenarios],
        counts,
        color=PALETTE["neutral"],
        edgecolor="white",
        linewidth=0.8,
        width=0.6,
    )
    ax.set_title("Synthetic Dataset — Scenario Type Distribution", fontsize=13, pad=12)
    ax.set_ylabel("Number of logs")
    ax.set_ylim(0, max(counts) * 1.25)
    for bar, count in zip(bars, counts):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            str(count),
            ha="center", va="bottom", fontsize=10, fontweight="bold",
        )
    ax.axhline(y=20, color=PALETTE["dark"], linewidth=0.8, linestyle="--", alpha=0.5,
               label="Expected (balanced, n=20)")
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig1_scenario_distribution.png", bbox_inches="tight")
    plt.show()
    print(f"\nSaved → {FIG_DIR / 'fig1_scenario_distribution.png'}")

---
## Section 3 — Annotation Pipeline

`src/annotation/annotator_simulator.py` simulates five annotator personas, each with systematic scoring biases. This is not an approximation — it is the design: structured bias produces measurable disagreement, which is the ground truth for measuring calibration effectiveness.

| Persona | Primary Bias |
|---|---|
| **PrivacyMaximalist** | Suppresses `privacy_compliance` scores; strictest consent enforcement |
| **OutcomeOptimist** | Inflates `goal_alignment`; goal-achievement focus |
| **ProcessPurist** | Strict on `step_quality`; penalizes reasoning shortcuts |
| **ClinicalSafetyFirst** | High `goal_alignment` for health alerts, low for others |
| **RecoverySkeptic** | Suppresses `error_recovery`; high bar for resilience |

All scores are integers on a 1–4 scale across four dimensions: `step_quality`, `privacy_compliance`, `goal_alignment`, `error_recovery`.

In [ ]:
"""Load annotation records and show sample + persona distribution."""
ann_path = DATA_ANN / "day12_annotations.jsonl"

if not ann_path.exists():
    print(f"TODO: Run annotator simulator first.\n"
          f"Run: uv run python -m src.annotation.annotator_simulator "
          f"--input {DATA_RAW / 'synthetic_wearable_logs.jsonl'} "
          f"--output {ann_path} --n-trajectories 5 --dry-run")
    annotations: list[dict] = []
else:
    annotations = [
        json.loads(line)
        for line in ann_path.read_text().splitlines()
        if line.strip()
    ]
    print(f"Loaded {len(annotations)} annotation records.")

    sample_ann = annotations[0]
    print(f"\n── Sample annotation record ──")
    for key in ("annotation_id", "log_id", "persona_name", "scenario_type",
                "step_quality", "privacy_compliance", "goal_alignment", "error_recovery"):
        print(f"  {key:<25}: {sample_ann.get(key)}")
    print(f"  {'rationale':<25}: {sample_ann.get('rationale', '')[:80]}...")

In [ ]:
"""Chart: mean scores per annotator persona, highlighting systematic bias."""
if not annotations:
    print("No annotation data — skipping chart.")
else:
    dimensions = ["step_quality", "privacy_compliance", "goal_alignment", "error_recovery"]
    personas = sorted({r["persona_name"] for r in annotations})

    # Compute per-persona mean per dimension
    persona_means: dict[str, dict[str, float]] = {}
    for persona in personas:
        recs = [r for r in annotations if r["persona_name"] == persona]
        persona_means[persona] = {
            dim: sum(r[dim] for r in recs) / len(recs)
            for dim in dimensions
        }

    x = np.arange(len(dimensions))
    width = 0.15
    colors = ["#3498db", "#2ecc71", "#e67e22", "#e74c3c", "#9b59b6"]

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (persona, color) in enumerate(zip(personas, colors)):
        means = [persona_means[persona][dim] for dim in dimensions]
        offset = (i - len(personas) / 2 + 0.5) * width
        ax.bar(x + offset, means, width, label=persona, color=color, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels([d.replace("_", "\n") for d in dimensions])
    ax.set_ylabel("Mean score (1–4 scale)")
    ax.set_title("Annotator Persona Biases — Mean Score per Dimension", fontsize=13, pad=12)
    ax.set_ylim(1, 4.5)
    ax.axhline(y=2.5, color=PALETTE["dark"], linewidth=0.8, linestyle="--", alpha=0.4,
               label="Scale midpoint (2.5)")
    ax.legend(fontsize=8, frameon=False, ncol=3, loc="upper right")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig2_persona_bias.png", bbox_inches="tight")
    plt.show()
    print(f"\nSaved → {FIG_DIR / 'fig2_persona_bias.png'}")

---
## Section 4 — Inter-Annotator Agreement (IRR/IAA)

We measure agreement at two levels:

**Standard IRR (path-comparison):** Fleiss' κ on the raw step-level labels across all annotators. Negative κ is expected — annotators seeing different-length paths to the same goal appear to disagree even when they agree on *quality*.

**Path-Invariant Agreement (PIA):** Our original methodological contribution. Instead of comparing step choices, we compare rubric-dimension scores (planning quality, error recovery, goal alignment) evaluated independently of path length. This resolves the non-determinism paradox for agentic systems.

### Calibration Protocol  
5 anchor examples (2 clearly_good, 1 borderline, 2 clearly_bad) with gold scores. Calibration weight = **0.82**: `calibrated = 0.82 × gold + 0.18 × persona_base`. Target: κ ≥ 0.55, α ≥ 0.80.

> **Caveat:** Post-calibration κ = 1.0 in all figures below is a **mathematical artifact of deterministic dry-run blending** — the formula collapses all scores to the same values. Live-API annotation (real Anthropic API calls) is expected to yield Cohen's κ ≈ 0.55–0.65 pre → 0.78–0.85 post. The pre-calibration numbers are real.

In [ ]:
"""Load IRR results from calibration data file."""
cal_path = DATA_ANN / "post_calibration" / "annotations_round2.json"
pia_path = DATA_ANN / "pia_results.json"

irr_data: dict = {}
pia_data: dict = {}

if not cal_path.exists():
    print(f"TODO: Missing {cal_path}")
    print("Run: uv run python -m src.annotation.run_calibrated_annotation --dry-run")
else:
    cal = json.loads(cal_path.read_text())
    irr_data = cal["irr_results"]
    meta = cal["metadata"]
    print(f"Calibration metadata:")
    print(f"  Round             : {meta['round']}")
    print(f"  Trajectories      : {meta['n_logs']}")
    print(f"  Records           : {meta['n_records']}  ({meta['n_logs']} logs × 5 personas)")
    print(f"  Calibration weight: {meta['calibration_weight']}")
    print(f"  Target α          : {meta['target_alpha']}")
    print(f"  Target κ          : {meta['target_kappa']}")

    pre = irr_data["pre_calibration"]
    print(f"\nPre-calibration Fleiss' κ by dimension:")
    dims = ["step_quality", "privacy_compliance", "goal_alignment", "error_recovery"]
    for dim in dims:
        k = pre[dim]["fleiss_kappa"]
        interp = pre[dim]["fleiss_interpretation"]
        print(f"  {dim:<25}: {k:+.4f}  ({interp})")
    print(f"  {'overall':<25}: {pre['overall']['fleiss_kappa']:+.4f}  ({pre['overall']['fleiss_interpretation']})")

if not pia_path.exists():
    print(f"\nTODO: Missing {pia_path}")
    print("Run: uv run python -m src.annotation.pia_calculator --dry-run")
else:
    pia_data = json.loads(pia_path.read_text())
    print(f"\nPIA results:")
    print(f"  Standard κ (path-comparison): {pia_data['standard_overall_kappa']:+.4f}  ({pia_data['standard_interpretation']})")
    print(f"  PIA κ    (rubric-dimension):  {pia_data['pia_overall_kappa']:+.4f}  ({pia_data['pia_interpretation']})")
    print(f"  Δκ       (improvement):       +{pia_data['delta']:.4f}")
    print(f"  → {pia_data['delta_headline']}")

In [ ]:
"""Chart A: Pre-calibration Fleiss' κ by annotation dimension."""
if not irr_data:
    print("No IRR data — skipping chart.")
else:
    dims = ["step_quality", "privacy_compliance", "goal_alignment", "error_recovery"]
    pre = irr_data["pre_calibration"]
    pre_vals = [pre[d]["fleiss_kappa"] for d in dims]

    # Post values (dry-run artifact — shown with hatching and explicit label)
    post = irr_data["post_calibration"]
    post_vals = [post[d]["fleiss_kappa"] for d in dims]

    x = np.arange(len(dims))
    width = 0.35
    dim_labels = [d.replace("_", "\n") for d in dims]

    fig, ax = plt.subplots(figsize=(10, 5))

    bars_pre = ax.bar(
        x - width / 2, pre_vals, width,
        label="Pre-calibration (real)",
        color=PALETTE["bad"], alpha=0.85,
    )
    bars_post = ax.bar(
        x + width / 2, post_vals, width,
        label="Post-calibration (dry-run artifact — not citable)",
        color=PALETTE["good"], alpha=0.4,
        hatch="//",
    )

    ax.axhline(y=0, color="black", linewidth=0.8, linestyle="-")
    ax.axhline(y=0.55, color=PALETTE["neutral"], linewidth=1.2, linestyle="--",
               alpha=0.7, label="Target κ = 0.55")
    ax.axhline(y=0.80, color=PALETTE["good"], linewidth=1.2, linestyle=":",
               alpha=0.8, label="Target α = 0.80")

    for bar, val in zip(bars_pre, pre_vals):
        ypos = bar.get_height() - 0.04 if val < 0 else bar.get_height() + 0.02
        ax.text(
            bar.get_x() + bar.get_width() / 2, ypos,
            f"{val:+.3f}", ha="center", va="top" if val < 0 else "bottom",
            fontsize=9, fontweight="bold", color=PALETTE["bad"],
        )

    ax.set_xticks(x)
    ax.set_xticklabels(dim_labels)
    ax.set_ylabel("Fleiss' κ")
    ax.set_title(
        "IAA Before & After Calibration — Fleiss' κ by Annotation Dimension",
        fontsize=13, pad=12,
    )
    ax.set_ylim(-0.35, 1.25)
    ax.legend(fontsize=9, frameon=False, loc="upper left")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig3_iaa_before_after.png", bbox_inches="tight")
    plt.show()
    print(f"\nSaved → {FIG_DIR / 'fig3_iaa_before_after.png'}")

    # Key finding
    below = sum(1 for v in pre_vals if v < 0.6)
    print(f"\n▶ Key finding: {below}/{len(dims)} dimensions fall below κ = 0.60 pre-calibration.")
    print(f"  All 4 are in 'poor' territory (κ < 0). Addresses Cohere Command A gap:")
    print(f"  800-prompt annotation with zero reported agreement statistics.")

In [ ]:
"""Chart B: Standard κ vs PIA κ — the path-invariant agreement improvement."""
if not pia_data:
    print("No PIA data — skipping chart.")
else:
    pia_dims = ["planning_quality", "error_recovery", "goal_alignment"]
    pia_dim_kappas = [pia_data["pia_per_dimension_kappa"][d] for d in pia_dims]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left: Standard vs PIA overall comparison
    ax_left = axes[0]
    methods = ["Standard\n(path-comparison)", "PIA\n(rubric-dimension)"]
    kappas = [pia_data["standard_overall_kappa"], pia_data["pia_overall_kappa"]]
    bar_colors = [PALETTE["bad"], PALETTE["good"]]

    bars = ax_left.bar(methods, kappas, color=bar_colors, alpha=0.85, width=0.45)
    ax_left.axhline(y=0, color="black", linewidth=0.8)
    ax_left.axhline(y=0.61, color="gray", linewidth=0.8, linestyle="--",
                    label="Substantial threshold (0.61)")
    for bar, val in zip(bars, kappas):
        ypos = val + 0.03 if val >= 0 else val - 0.06
        ax_left.text(
            bar.get_x() + bar.get_width() / 2, ypos,
            f"κ = {val:+.3f}", ha="center", fontsize=11, fontweight="bold",
        )
    ax_left.annotate(
        f"Δ = +{pia_data['delta']:.3f}",
        xy=(0.5, 0.5),
        xycoords="axes fraction",
        ha="center", fontsize=12,
        fontweight="bold", color=PALETTE["neutral"],
    )
    ax_left.set_ylabel("Fleiss' κ")
    ax_left.set_title("Standard IRR vs Path-Invariant Agreement", fontsize=12)
    ax_left.set_ylim(-0.2, 1.0)
    ax_left.legend(fontsize=8, frameon=False)

    # Right: PIA κ by rubric dimension
    ax_right = axes[1]
    bar_cols_dims = [PALETTE["neutral"]] * len(pia_dims)
    bars2 = ax_right.bar(
        [d.replace("_", "\n") for d in pia_dims],
        pia_dim_kappas,
        color=bar_cols_dims, alpha=0.85, width=0.5,
    )
    ax_right.axhline(y=0.61, color="gray", linewidth=0.8, linestyle="--",
                     label="Substantial (0.61)")
    for bar, val in zip(bars2, pia_dim_kappas):
        ax_right.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{val:.3f}", ha="center", fontsize=10, fontweight="bold",
        )
    ax_right.set_ylabel("PIA Fleiss' κ")
    ax_right.set_title("PIA κ by Rubric Dimension", fontsize=12)
    ax_right.set_ylim(0, 1.1)
    ax_right.legend(fontsize=8, frameon=False)

    fig.suptitle(
        "Path-Invariant Agreement (PIA) — Original Methodological Contribution",
        fontsize=13, y=1.01,
    )
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig4_pia_vs_standard.png", bbox_inches="tight")
    plt.show()
    print(f"\nSaved → {FIG_DIR / 'fig4_pia_vs_standard.png'}")
    print(f"\n▶ Key finding: PIA raises overall κ from {pia_data['standard_overall_kappa']:+.3f} "
          f"to {pia_data['pia_overall_kappa']:+.3f} (Δ = +{pia_data['delta']:.3f}).")
    print(f"  privacy_sensitive scenario reaches PIA κ = 0.942 — near-perfect agreement")
    print(f"  on privacy dimension once path length is factored out.")

---
## Section 5 — PRM Labeling & the Gradient Conflict Problem

**The gradient conflict problem** (ReasonRAG, NeurIPS 2025): an Outcome Reward Model (ORM) assigns reward only at the terminal step. If the terminal step fails, the ORM propagates −1 reward to *every* intermediate step — including steps that were individually correct. This destroys step-level signal that PRM would preserve.

We implement a **Process Reward Model (PRM)** via `src/annotation/prm_annotator.py`. Each step receives:
- `process_reward_score` ∈ [−1, +1]: quality of the individual step decision
- `partial_credit` ∈ [0, 1]: fractional credit even in failed trajectories  
- `outcome_reward`: non-zero **only** at the terminal step (±1)

A trajectory is a **gradient conflict** when it has `outcome_success=False` AND ≥ 50% of non-terminal steps have `process_reward_score > 0`. These are the trajectories that ORM penalizes despite containing correct decision-making.

In [ ]:
"""Load PRM annotations and summary statistics."""
prm_path = DATA_ANN / "prm_annotations_20.jsonl"
prm_stats_path = DATA_ANN / "prm_summary_stats.json"

prm_records: list[dict] = []
prm_stats: dict = {}

if not prm_stats_path.exists():
    print(f"TODO: Missing {prm_stats_path}")
    print("Run: uv run python -m src.annotation.prm_annotator "
          "--input data/raw/synthetic_wearable_logs.jsonl "
          "--output data/annotations/prm_annotations_20.jsonl "
          "--limit 20 --summary-output data/annotations/prm_summary_stats.json")
else:
    prm_stats = json.loads(prm_stats_path.read_text())
    print("PRM summary statistics:")
    for key, val in prm_stats.items():
        pct = f"  ({val*100:.1f}%)" if isinstance(val, float) and 0 < val <= 1.0 else ""
        print(f"  {key:<45}: {val}{pct}")

if prm_path.exists():
    prm_records = [
        json.loads(line)
        for line in prm_path.read_text().splitlines()
        if line.strip()
    ]
    print(f"\nLoaded {len(prm_records)} PRM-annotated trajectories.")

    # Flatten to step level
    all_steps = [
        {**step, "outcome_success": rec["outcome_success"], "gradient_conflict": rec["gradient_conflict"]}
        for rec in prm_records
        for step in rec["steps"]
    ]
    non_terminal = [s for s in all_steps if not s["is_terminal"]]
    terminal = [s for s in all_steps if s["is_terminal"]]
    print(f"  Total steps         : {len(all_steps)}")
    print(f"  Non-terminal steps  : {len(non_terminal)}")
    print(f"  Terminal steps      : {len(terminal)}")

In [ ]:
"""Chart: PRM step-reward distribution + gradient conflict visualization."""
if not prm_records or not prm_stats:
    print("No PRM data — skipping chart.")
else:
    all_steps = [
        {**step, "outcome_success": rec["outcome_success"], "gradient_conflict": rec["gradient_conflict"]}
        for rec in prm_records
        for step in rec["steps"]
    ]
    non_terminal = [s for s in all_steps if not s["is_terminal"]]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left: process_reward_score distribution for non-terminal steps
    ax_left = axes[0]
    prs_values = [s["process_reward_score"] for s in non_terminal]
    ax_left.hist(
        prs_values, bins=20, range=(-1.0, 1.0),
        color=PALETTE["neutral"], alpha=0.8, edgecolor="white",
    )
    ax_left.axvline(x=0, color="black", linewidth=1.0, linestyle="-")
    ax_left.axvline(
        x=np.mean(prs_values), color=PALETTE["warn"],
        linewidth=1.5, linestyle="--",
        label=f"Mean = {np.mean(prs_values):.3f}",
    )
    ax_left.set_xlabel("Process Reward Score (PRS)")
    ax_left.set_ylabel("Step count")
    ax_left.set_title("PRS Distribution — Non-Terminal Steps", fontsize=12)
    ax_left.legend(fontsize=9, frameon=False)

    # Right: gradient conflict summary (pie / annotated bar)
    ax_right = axes[1]
    conflict_count = prm_stats["gradient_conflict_count"]
    total = prm_stats["total_trajectories"]
    no_conflict = total - conflict_count

    wedge_colors = [PALETTE["bad"], PALETTE["good"]]
    wedge_labels = [
        f"Gradient conflict\n({conflict_count}/{total} trajectories)",
        f"No conflict\n({no_conflict}/{total} trajectories)",
    ]
    wedge_sizes = [conflict_count, no_conflict]

    # Handle 100% conflict (no_conflict=0 → pie becomes degenerate)
    if no_conflict == 0:
        ax_right.bar(
            ["Gradient Conflict", "No Conflict"],
            [conflict_count, no_conflict],
            color=[PALETTE["bad"], PALETTE["good"]],
            alpha=0.85, width=0.5,
        )
        ax_right.set_ylabel("Trajectory count")
        ax_right.set_title("Gradient Conflict Rate", fontsize=12)
        ax_right.text(
            0, conflict_count / 2,
            f"100%\n({conflict_count} trajectories)",
            ha="center", va="center",
            fontsize=13, fontweight="bold", color="white",
        )
    else:
        wedges, texts, autotexts = ax_right.pie(
            wedge_sizes, labels=wedge_labels, colors=wedge_colors,
            autopct="%1.0f%%", startangle=90,
            pctdistance=0.6, textprops={"fontsize": 10},
        )
        for autotext in autotexts:
            autotext.set_fontweight("bold")
    ax_right.set_title(
        "Gradient Conflict Rate\n"
        "(outcome-failed trajectories with ≥50% correct steps)",
        fontsize=11,
    )

    fig.suptitle(
        "Process Reward Model (PRM) — Step-Level Annotation Results",
        fontsize=13, y=1.02,
    )
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig5_prm_gradient_conflict.png", bbox_inches="tight")
    plt.show()
    print(f"\nSaved → {FIG_DIR / 'fig5_prm_gradient_conflict.png'}")
    print(f"\n▶ Key finding: {prm_stats['gradient_conflict_rate']*100:.0f}% gradient conflict rate.")
    print(f"  (Synthetic artifact: all trajectories initialized with outcome_success=False.)")
    print(f"  Demonstrates the ORM failure mode: −1 reward would propagate to")
    print(f"  {prm_stats['mean_process_reward_non_terminal']:.3f} mean PRS across correct intermediate steps.")

---
## Section 6 — Annotator Poisoning Detection

Inspired by Anthropic's October 2025 finding that **250 documents** (count-based, not proportion-based) suffice to backdoor a model of any size, we test whether our annotation pipeline can detect malicious annotators.

**Detection approach:** MAD (Median Absolute Deviation) suspicion scoring. Each annotator receives a suspicion score ∈ [0, 1] based on deviation from panel consensus. High score = outlier.

**Named blind spot (WP1 empirical finding):** Three colluding annotators with *identical* bias collapse to panel consensus under MAD scoring. Their shared deviation averages to zero, yielding suspicion = 0.0 — the same score as the most reliable annotator. This is not a bug; it is a fundamental property of MAD-based detection.

**Second-layer defense:** Cleanlab Confident Learning (`find_label_issues` + `get_label_quality_scores`) as a label noise detector, independent of the annotator-level signal.

In [ ]:
"""Load poisoning detection results."""
detection_path = DATA_PROC / "day17_detection_results.json"

detect_data: dict = {}

if not detection_path.exists():
    print(f"TODO: Missing {detection_path}")
    print("Run: uv run python scripts/run_day17_detection.py")
else:
    detect_data = json.loads(detection_path.read_text())
    print(f"Detection results:")
    print(f"  Input records  : {detect_data['n_records_loaded']} (5 logs × 5 personas)")
    print(f"  After injection: {detect_data['n_records_augmented']} (+ 3 synthetic poisoners × 5 logs)")
    print(f"\nClean pool MAD suspicion scores:")
    for name, score in detect_data["clean_pool_scores"].items():
        flag = "  ← lowest" if score == 0.0 else ("  ← highest" if score == 1.0 else "")
        print(f"  {name:<25}: {score:.4f}{flag}")
    print(f"\nAugmented pool (with 3 injected poisoners):")
    for name, score in detect_data["augmented_pool_scores"].items():
        is_poisoner = "Poisoner" in name
        tag = "  ◄ INJECTED POISONER" if is_poisoner else ""
        print(f"  {name:<25}: {score:.4f}{tag}")
    ev = detect_data["evaluation"]
    print(f"\nDetection evaluation (threshold = {ev['threshold']}):")
    print(f"  TP = {ev['true_positives']}, FP = {ev['false_positives']}, FN = {ev['false_negatives']}")
    print(f"  Precision = {ev['precision']:.3f} | Recall = {ev['recall']:.3f} | F1 = {ev['f1']:.3f}")
    print(f"\n▶ MAD blind spot confirmed: all 3 colluding poisoners score 0.0 (suspicion = lowest).")

In [ ]:
"""Chart: Annotator suspicion scores + ROC curve from threshold sweep."""
if not detect_data:
    print("No detection data — skipping chart.")
else:
    augmented_scores = detect_data["evaluation"]["per_annotator_scores"]
    poisoner_names = {"Poisoner_A", "Poisoner_B", "Poisoner_C"}

    names = list(augmented_scores.keys())
    scores = [augmented_scores[n] for n in names]
    bar_colors = [
        PALETTE["bad"] if n in poisoner_names else PALETTE["neutral"]
        for n in names
    ]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # ── Left: suspicion score bar chart ────────────────────────────────────
    ax_left = axes[0]
    display_names = [
        n.replace("Poisoner_", "Poisoner\n") for n in names
    ]
    bars = ax_left.bar(
        display_names, scores,
        color=bar_colors, alpha=0.85,
        edgecolor="white", linewidth=0.8,
    )
    ax_left.axhline(
        y=0.6, color=PALETTE["warn"], linewidth=1.5,
        linestyle="--", label="Detection threshold (0.6)",
    )
    for bar, val in zip(bars, scores):
        ax_left.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{val:.3f}", ha="center", va="bottom", fontsize=8,
        )
    from matplotlib.patches import Patch
    legend_els = [
        Patch(facecolor=PALETTE["bad"], label="Injected poisoner"),
        Patch(facecolor=PALETTE["neutral"], label="Legitimate annotator"),
    ]
    ax_left.legend(handles=legend_els, fontsize=9, frameon=False)
    ax_left.set_ylabel("MAD suspicion score [0, 1]")
    ax_left.set_title("Annotator Suspicion Scores\n(MAD-based, threshold = 0.6)", fontsize=11)
    ax_left.set_ylim(0, 1.25)
    plt.setp(ax_left.get_xticklabels(), fontsize=8)

    # ── Right: ROC curve from threshold sweep ─────────────────────────────
    ax_right = axes[1]

    n_pos = len(poisoner_names)          # 3
    n_neg = len(names) - n_pos           # 5
    thresholds = np.linspace(0.0, 1.01, 500)

    tpr_list = []
    fpr_list = []
    for thresh in thresholds:
        flagged = {n for n, s in augmented_scores.items() if s >= thresh}
        tp = len(flagged & poisoner_names)
        fp = len(flagged - poisoner_names)
        tpr = tp / n_pos if n_pos > 0 else 0.0
        fpr = fp / n_neg if n_neg > 0 else 0.0
        tpr_list.append(tpr)
        fpr_list.append(fpr)

    # AUC via trapezoidal rule (reverse so fpr is monotonically increasing)
    fpr_arr = np.array(fpr_list[::-1])
    tpr_arr = np.array(tpr_list[::-1])
    auc = float(np.trapezoid(tpr_arr, fpr_arr))

    ax_right.plot(
        fpr_list, tpr_list,
        color=PALETTE["bad"], linewidth=2.0,
        label=f"MAD detector (AUC = {auc:.3f})",
    )
    ax_right.plot(
        [0, 1], [0, 1],
        color="gray", linewidth=1.0, linestyle="--",
        label="Random baseline (AUC = 0.500)",
    )
    # Annotate the blind spot
    ax_right.annotate(
        "MAD blind spot:\n3 colluding poisoners\nscore 0.0 (not flagged)",
        xy=(1.0, 0.0), xytext=(0.55, 0.15),
        fontsize=8, color=PALETTE["bad"],
        arrowprops={"arrowstyle": "->", "color": PALETTE["bad"]},
    )
    ax_right.set_xlabel("False Positive Rate")
    ax_right.set_ylabel("True Positive Rate (Recall)")
    ax_right.set_title("Poisoning Detector ROC Curve\n(threshold sweep 0.0 → 1.0)", fontsize=11)
    ax_right.set_xlim(-0.02, 1.05)
    ax_right.set_ylim(-0.02, 1.05)
    ax_right.legend(fontsize=9, frameon=False, loc="upper left")

    fig.suptitle(
        "Annotator Poisoning Detection — MAD Detector & Blind Spot Analysis",
        fontsize=13, y=1.02,
    )
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig6_poisoning_detection.png", bbox_inches="tight")
    plt.show()
    print(f"\nSaved → {FIG_DIR / 'fig6_poisoning_detection.png'}")
    print(f"\n▶ ROC AUC = {auc:.3f} (MAD detector performs worse than random for colluding attackers).")
    print(f"  The degenerate curve is the finding — motivates cleanlab second-layer defense.")
    cl = detect_data["cleanlab"]
    print(f"\n▶ Cleanlab Confident Learning:")
    print(f"  Dimension: {cl['dimension']} | Logs examined: {cl['n_logs']} | Issues found: {cl['n_issues_found']}")

---
## Section 7 — Pipeline Summary

All six pipeline stages, their inputs/outputs, and headline metrics.

In [ ]:
"""Print the pipeline summary table and generate a visual version."""
summary_rows = [
    {
        "Stage": "1 · Synthetic Generation",
        "Module": "src/data/wearable_generator.py",
        "Input": "seed=42, count=100",
        "Output": "100 wearable logs (JSONL)",
        "Key Metric": "5 scenario types × 20 logs (balanced)",
    },
    {
        "Stage": "2 · Differential Privacy",
        "Module": "src/data/privacy_gate.py",
        "Input": "Raw sensor values",
        "Output": "DP-noised sensor fields",
        "Key Metric": "ε=1.0, σ≈48 bpm heart rate",
    },
    {
        "Stage": "3 · Multi-Persona Annotation",
        "Module": "src/annotation/annotator_simulator.py",
        "Input": "100 logs, 5 personas",
        "Output": "150 records (30 logs × 5 personas)",
        "Key Metric": "Pre-cal Fleiss' κ = −0.035 (poor)",
    },
    {
        "Stage": "4 · IAA Calibration + PIA",
        "Module": "src/annotation/pia_calculator.py",
        "Input": "150 annotation records",
        "Output": "pia_results.json",
        "Key Metric": "Standard κ −0.065 → PIA κ +0.743 (Δ=+0.808)",
    },
    {
        "Stage": "5 · PRM Step Labeling",
        "Module": "src/annotation/prm_annotator.py",
        "Input": "20 trajectories",
        "Output": "60 step rewards (JSONL)",
        "Key Metric": "100% gradient conflict rate (synthetic)",
    },
    {
        "Stage": "6 · Poisoning Detection",
        "Module": "src/annotation/poisoning_detector.py",
        "Input": "25 records + 3 injected poisoners",
        "Output": "day17_detection_results.json",
        "Key Metric": "MAD F1=0.0 (blind spot); Cleanlab second layer",
    },
]

col_widths = {col: max(len(col), max(len(str(row[col])) for row in summary_rows))
              for col in ["Stage", "Module", "Input", "Output", "Key Metric"]}

header = " | ".join(col.ljust(col_widths[col]) for col in col_widths)
sep = "-+-".join("-" * w for w in col_widths.values())
print(header)
print(sep)
for row in summary_rows:
    line = " | ".join(str(row[col]).ljust(col_widths[col]) for col in col_widths)
    print(line)

In [ ]:
"""Visual summary: headline metric panel — one number per pipeline stage."""
headline_metrics = [
    {"stage": "Synthetic\nData",       "value": "100",     "unit": "logs",      "color": PALETTE["neutral"]},
    {"stage": "DP Privacy\n(ε=1.0)",   "value": "~48",     "unit": "bpm σ",     "color": PALETTE["neutral"]},
    {"stage": "Pre-Cal\nFleiss' κ",    "value": "−0.035",  "unit": "(poor)",    "color": PALETTE["bad"]},
    {"stage": "PIA Δκ",               "value": "+0.808",  "unit": "κ points",  "color": PALETTE["good"]},
    {"stage": "Gradient\nConflict",   "value": "100%",    "unit": "(synthetic)","color": PALETTE["warn"]},
    {"stage": "MAD\nDetector F1",     "value": "0.0",     "unit": "(blind spot)","color": PALETTE["bad"]},
]

fig, axes = plt.subplots(1, len(headline_metrics), figsize=(14, 3))
for ax, m in zip(axes, headline_metrics):
    ax.set_facecolor(m["color"])
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.text(
        0.5, 0.62, m["value"],
        transform=ax.transAxes,
        ha="center", va="center",
        fontsize=20, fontweight="bold", color="white",
    )
    ax.text(
        0.5, 0.38, m["unit"],
        transform=ax.transAxes,
        ha="center", va="center",
        fontsize=9, color="white", alpha=0.9,
    )
    ax.set_xlabel(m["stage"], fontsize=10, labelpad=6)
    ax.xaxis.label.set_color(PALETTE["dark"])

fig.suptitle(
    "Privacy-First Curation Pipeline — Headline Results",
    fontsize=13, y=1.08,
)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig7_headline_metrics.png", bbox_inches="tight")
plt.show()
print(f"\nSaved → {FIG_DIR / 'fig7_headline_metrics.png'}")

---
## Appendix — Numbers Citability Reference

| Statistic | Value | Source File | Citable? |
|---|---|---|---|
| PIA Δκ | +0.808 | `data/annotations/pia_results.json` | ✅ Yes |
| Standard overall Fleiss' κ | −0.065 | `data/annotations/pia_results.json` | ✅ Yes |
| Pre-calibration Fleiss' κ overall | −0.035 | `annotations_round2.json → irr_results.pre_calibration.overall` | ✅ Yes |
| Pre-cal Fleiss' κ per dimension | −0.043 to +0.002 | `annotations_round2.json` | ✅ Yes |
| Gradient conflict rate | 100% | `prm_summary_stats.json` | ✅ with caveat (synthetic) |
| Mean PRS (non-terminal) | 0.175 | `prm_summary_stats.json` | ✅ with caveat |
| MAD detector F1 (3 colluders) | 0.0 | `day17_detection_results.json` | ✅ Yes (finding IS F1=0.0) |
| Post-calibration Fleiss' κ | 1.0 | `annotations_round2.json` | ❌ Dry-run artifact |
| Post-calibration Krippendorff α | 1.0 | `annotations_round2.json` | ❌ Dry-run artifact |
| Benchmark token counts | langgraph 460 → autogen 950 | `benchmark_results.jsonl` | ✅ Relative ordering only |

**Next steps (Day 18 → Day 28):**
- Run annotator simulator with live Anthropic API (HF_TOKEN + ANTHROPIC_API_KEY) to produce real post-calibration κ values
- Draft WP1 §1 (ORM→PRM paradigm shift) and §2 (why existing datasets fail step-level quality checks) in `white_papers/wp1_data_curation.md`
- HH-RLHF analysis for WP1 §6 (pending — no output files exist yet)